# Tokenization - Production Implementation

**Complete guide with real HuggingFace libraries and production patterns.**

This notebook uses:
- Real models from HuggingFace Hub
- Production-grade patterns
- Error handling and optimization
- Real-world use cases

See also: `llm/concepts/tokenization.md` for theory and interview Q&A

## Setup

In [ ]:
# Install required packages
# !pip install transformers torch sentence-transformers datasets peft bitsandbytes

import warnings
warnings.filterwarnings('ignore')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Quick Start

In [ ]:
# Tokenization with HuggingFace
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hello, how are you?"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"IDs: {token_ids}")
print(f"Decoded: {tokenizer.decode(token_ids)}")

## Production Implementation

In [ ]:
# Production Tokenization Pipeline
from transformers import AutoTokenizer
import torch

class TokenizationService:
    """Production-grade tokenization"""

    def __init__(self, model_name="bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_batch(self, texts, max_length=512, batch_size=32):
        """Batch tokenization with padding"""
        encodings = self.tokenizer(
            texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            batch_size=batch_size
        )
        return encodings

    def get_special_tokens(self):
        """Get special token IDs"""
        return {
            "sos": self.tokenizer.bos_token_id,
            "eos": self.tokenizer.eos_token_id,
            "pad": self.tokenizer.pad_token_id,
            "unk": self.tokenizer.unk_token_id
        }

# Usage
service = TokenizationService()
texts = ["Sample text 1", "Sample text 2"]
encodings = service.tokenize_batch(texts, max_length=128)
print(f"Shape: {encodings['input_ids'].shape}")

## Real-World: Streaming

In [ ]:
# Real-World: Streaming Tokenization
from transformers import AutoTokenizer
import torch

class StreamingTokenizer:
    """Tokenize streaming input efficiently"""

    def __init__(self, model_name="gpt2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.buffer = ""

    def process_chunk(self, text_chunk):
        """Process streaming text"""
        self.buffer += text_chunk

        # Try to tokenize complete words
        last_space = self.buffer.rfind(" ")

        if last_space > 0:
            to_tokenize = self.buffer[:last_space]
            self.buffer = self.buffer[last_space+1:]

            token_ids = self.tokenizer.encode(to_tokenize)
            return token_ids
        return []

    def flush(self):
        """Get remaining tokens"""
        if self.buffer:
            token_ids = self.tokenizer.encode(self.buffer)
            self.buffer = ""
            return token_ids
        return []

# Usage for streaming
streamer = StreamingTokenizer()
stream = ["Hello ", "world ", "this ", "is ", "streaming"]

all_tokens = []
for chunk in stream:
    tokens = streamer.process_chunk(chunk)
    all_tokens.extend(tokens)

final_tokens = streamer.flush()
all_tokens.extend(final_tokens)

print(f"Total tokens: {len(all_tokens)}")

## Production Checklist

- [ ] Load models from HuggingFace Hub
- [ ] Set up GPU device handling
- [ ] Implement batch processing
- [ ] Add error handling
- [ ] Optimize for latency
- [ ] Add logging and monitoring
- [ ] Test with production data
- [ ] Create inference service

## Useful Links

- [HuggingFace Models](https://huggingface.co/models)
- [HuggingFace Documentation](https://huggingface.co/docs/transformers)
- [PEFT Library](https://github.com/huggingface/peft)
- [Sentence Transformers](https://www.sbert.net/)